In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


# Paths

base = Path("../../data/full")

def pick_existing(candidates, name):
    fp = next((p for p in candidates if p.exists()), None)
    assert fp is not None, f"{name} not found. Checked: {candidates}"
    return fp

def read_csv_safe(fp):
    try:
        return pd.read_csv(fp)
    except Exception:
        return pd.read_csv(fp, engine="python", on_bad_lines="skip")

# raw vs clean candidates (exo raw kept as clean baseline to avoid malformed exoplanets.csv)
stars_raw_fp    = pick_existing([base / "stars_gaia.csv", base / "stars.csv"], "Stars raw file")
stars_clean_fp  = pick_existing([base / "stars_gaia_clean.csv", base / "stars_clean.csv"], "Stars clean file")

quasars_raw_fp   = pick_existing([base / "quasars_gaia.csv", base / "quasars.csv"], "Quasars raw file")
quasars_clean_fp = pick_existing([base / "quasars_gaia_clean.csv", base / "quasars_clean.csv"], "Quasars clean file")

exo_raw_fp    = pick_existing([base / "exoplanets_clean.csv", base / "exoplanets_gaia_enriched.csv"], "Exoplanet raw/baseline file")
exo_clean_fp  = pick_existing([base / "exoplanets_gaia_enriched.csv", base / "exoplanets_clean.csv"], "Exoplanet clean/enriched file")

print("Using files:")
print("stars_raw   :", stars_raw_fp)
print("stars_clean :", stars_clean_fp)
print("quasars_raw :", quasars_raw_fp)
print("quasars_clean:", quasars_clean_fp)
print("exo_raw     :", exo_raw_fp)
print("exo_clean   :", exo_clean_fp)


# Load

stars_raw = read_csv_safe(stars_raw_fp)
stars_clean = read_csv_safe(stars_clean_fp)

quasars_raw = read_csv_safe(quasars_raw_fp)
quasars_clean = read_csv_safe(quasars_clean_fp)

exo_raw = read_csv_safe(exo_raw_fp)
exo_clean = read_csv_safe(exo_clean_fp)


# Shared features

features = ["ra", "dec", "parallax", "pmra", "pmdec", "phot_g_mean_mag"]

def align_exo_cols(df):
    rename_map = {
        "sy_plx": "parallax",
        "sy_pmra": "pmra",
        "sy_pmdec": "pmdec",
        "sy_gaiamag": "phot_g_mean_mag",
    }
    present = {k: v for k, v in rename_map.items() if k in df.columns}
    return df.rename(columns=present).copy()

def prep_for_shared_dropna(df, is_exo=False):
    d = align_exo_cols(df) if is_exo else df.copy()
    for c in features:
        if c not in d.columns:
            d[c] = np.nan
    d[features] = d[features].apply(pd.to_numeric, errors="coerce")
    return d

stars_p = prep_for_shared_dropna(stars_clean, is_exo=False)
quasars_p = prep_for_shared_dropna(quasars_clean, is_exo=False)
exo_p = prep_for_shared_dropna(exo_clean, is_exo=True)

def counts(raw_df, clean_df_p):
    rows_raw = len(raw_df)
    rows_after_script = len(clean_df_p)
    missing_cells_before = int(clean_df_p[features].isna().sum().sum())

    complete_df = clean_df_p.dropna(subset=features)
    rows_after_dropna = len(complete_df)
    missing_cells_after = int(complete_df[features].isna().sum().sum())

    removed_rows = rows_after_script - rows_after_dropna
    removed_pct = (removed_rows / rows_after_script * 100) if rows_after_script else 0.0

    return rows_raw, rows_after_script, rows_after_dropna, removed_rows, removed_pct, missing_cells_before, missing_cells_after

s = counts(stars_raw, stars_p)
q = counts(quasars_raw, quasars_p)
e = counts(exo_raw, exo_p)


# Table 3

table3_counts = pd.DataFrame([
    {"Dataset":"Stars", "Rows (raw)":s[0], "Rows (after script cleaning)":s[1], "Rows (after shared-feature dropna)":s[2]},
    {"Dataset":"Quasars", "Rows (raw)":q[0], "Rows (after script cleaning)":q[1], "Rows (after shared-feature dropna)":q[2]},
    {"Dataset":"Exoplanet hosts", "Rows (raw)":e[0], "Rows (after script cleaning)":e[1], "Rows (after shared-feature dropna)":e[2]},
])


# Table 5

table5_missing = pd.DataFrame([
    {"Dataset":"Stars", "Missing cells before":s[5], "Missing cells after":s[6], "Rows removed by dropna":s[3], "Rows removed (%)":round(s[4], 2)},
    {"Dataset":"Quasars", "Missing cells before":q[5], "Missing cells after":q[6], "Rows removed by dropna":q[3], "Rows removed (%)":round(q[4], 2)},
    {"Dataset":"Exoplanet hosts", "Missing cells before":e[5], "Missing cells after":e[6], "Rows removed by dropna":e[3], "Rows removed (%)":round(e[4], 2)},
])

print("\nTable 3 counts")
display(table3_counts)

print("Table 5 missing-handling")
display(table5_missing)




Using files:
stars_raw   : ..\..\data\full\stars_gaia.csv
stars_clean : ..\..\data\full\stars_gaia_clean.csv
quasars_raw : ..\..\data\full\quasars_gaia.csv
quasars_clean: ..\..\data\full\quasars_gaia_clean.csv
exo_raw     : ..\..\data\full\exoplanets_clean.csv
exo_clean   : ..\..\data\full\exoplanets_gaia_enriched.csv

Table 3 counts


,Dataset,Rows (raw),Rows (after script cleaning),Rows (after shared-feature dropna)
0,Stars,50000,50000,50000
1,Quasars,50000,50000,50000
2,Exoplanet hosts,37871,37871,0


Table 5 missing-handling


,Dataset,Missing cells before,Missing cells after,Rows removed by dropna,Rows removed (%)
0,Stars,0,0,0,0.0
1,Quasars,0,0,0,0.0
2,Exoplanet hosts,113613,0,37871,100.0



--- LaTeX: Table 3 ---
\begin{tabular}{lrrr}
\toprule
Dataset & Rows (raw) & Rows (after script cleaning) & Rows (after shared-feature dropna) \\
\midrule
Stars & 50000 & 50000 & 50000 \\
Quasars & 50000 & 50000 & 50000 \\
Exoplanet hosts & 37871 & 37871 & 0 \\
\bottomrule
\end{tabular}


--- LaTeX: Table 5 ---
\begin{tabular}{lrrrr}
\toprule
Dataset & Missing cells before & Missing cells after & Rows removed by dropna & Rows removed (%) \\
\midrule
Stars & 0 & 0 & 0 & 0.000000 \\
Quasars & 0 & 0 & 0 & 0.000000 \\
Exoplanet hosts & 113613 & 0 & 37871 & 100.000000 \\
\bottomrule
\end{tabular}

